# Notebook 04 – Feature Engineering

## Objective

### Transform raw customer, order, product, and website session data into a customer-level feature matrix suitable for customer segmentation.

### Input

#### - customers.csv
#### - orders.csv
#### - order_line_items.csv
#### - website_sessions.csv
#### - sku_catalog.csv

### Output

#### - customer_features.csv

### Each row in the final dataset represents one customer with engineered behavioral, transactional, and engagement features.

In [1]:
import numpy as np
import pandas as pd
from pathlib import Path
import warnings
warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", None)

In [2]:
PROCESSED = Path("data/processed")

customers = pd.read_csv(PROCESSED/"customers.csv")
orders = pd.read_csv(PROCESSED/"orders.csv")
order_items = pd.read_csv(PROCESSED/"order_items.csv")
website_sessions = pd.read_csv(PROCESSED/"website_sessions.csv")
sku_catalog = pd.read_csv(PROCESSED/"sku_catalog.csv")

### Standardize column names:

In [3]:
orders = orders.rename(columns={
    "order_date_&_time": "order_datetime",
    "order_value_(gross)": "gross_value",
    "order_value_(net)": "net_value",
    "discount_applied_(₹)": "discount_amount",
    "discount_applied_(%)": "discount_percent",
    "payment_mode": "payment_mode",
    "shipping_city": "shipping_city",
    "first_order_vs_repeat": "customer_type",
    "channel_source_(last_touch)": "channel",
    "delivered_/_returned_/_rto": "order_status"
})

order_items = order_items.rename(columns={
    "category_(top,_bottom,_outerwear,_etc.)": "category",
    "discount_%": "discount_percent",
    "returned?_(y/n)": "returned",
    "return_reason_(if_any)": "return_reason"
})

website_sessions = website_sessions.rename(columns={
    "date": "session_date"
})

sku_catalog = sku_catalog.rename(columns={
    "sku": "sku_id"
})

### Convert data columns:

In [4]:
orders["order_datetime"] = pd.to_datetime(
    orders["order_datetime"],
    errors="coerce"
)

website_sessions["session_date"] = pd.to_datetime(
    website_sessions["session_date"],
    errors="coerce"
)

### Create Master Transaction Dataset:

In [5]:
master = (
    orders
    .merge(order_items, on="order_id", how="left")
    .merge(
        sku_catalog[["sku_id", "vendor", "cost_per_unit"]],
        on="sku_id",
        how="left"
    )
)
print(master.shape)
master.head()

(42108, 27)


,order_id,customer_id,order_datetime,product,"order_value_(gross,_net)",gross_value,net_value,discount_applied_(₹_+_%),discount_amount,discount_percent_x,payment_mode,shipping_city,pincode,customer_type,channel,order_status,sku_id,category,size,color,mrp,selling_price,discount_percent_y,returned,return_reason,vendor,cost_per_unit
0,ORD-20220101-000001,CUST-0001627,2022-01-01 20:18:19,2-item cart,"2339,1814",2339,1814,"525,22.5%",525,22.5%,Card,Lucknow,624797,First,Meta,Delivered,SKU-PAN-0002,pants,XL,Maroon,2699,1814,32.8,N,NaN,Apex Fashions,982
1,ORD-20220101-000002,CUST-0006543,2022-01-01 23:42:40,2-item cart,"2588,1812",2588,1812,"777,30.0%",777,30.0%,UPI,Delhi,779192,First,Meta,Delivered,SKU-SHI-0001,shirts,XXL,Maroon,1999,1812,9.4,N,NaN,UrbanWeave Co.,705
2,ORD-20220101-000003,CUST-0021661,2022-01-01 22:25:30,2-item cart,"2546,2210",2546,2210,"336,13.2%",336,13.2%,UPI,Chandigarh,516352,First,Organic Instagram,Delivered,SKU-CRO-0019,croptops,L,Olive,1799,2210,0.0,N,NaN,StitchWorks India,726
3,ORD-20220101-000004,CUST-0007414,2022-01-01 10:16:28,2-item cart,"1800,1395",1800,1395,"405,22.5%",405,22.5%,COD,Indore,268472,First,Influencer,RTO,SKU-SHI-0003,shirts,L,Black,2799,1395,50.0,Y,RTO,KnitCraft Studio,1004
4,ORD-20220101-000005,CUST-0019325,2022-01-01 22:34:26,2-item cart,"2155,1853",2155,1853,"302,14.0%",302,14.0%,UPI,Mumbai,515911,First,Meta,Delivered,SKU-SHI-0045,shirts,XS,Maroon,2299,999,50.0,N,NaN,UrbanWeave Co.,896


### Customer base table:

In [6]:
customer_features = customers.copy()
customer_features.head()

,customer_id,name,first_order_date,total_orders,total_revenue,average_order_value,time_to_2nd_purchase,last_purchase_date,city_/_tier,acquisition_channel_(first_touch),repurchased
0,CUST-0000001,Tejas Sarkar,2025-09-08,1,3356.0,3356.0,NaN,2025-09-08,Mumbai / Tier 1,Meta,N
1,CUST-0000003,Lila Bedi,2023-09-11,1,2360.0,2360.0,NaN,2023-09-11,Bengaluru / Tier 1,Google,N
2,CUST-0000004,Hamsini Parekh,2022-03-18,2,4029.0,2014.5,449.0,2023-06-11,Hyderabad / Tier 1,Google,Y
3,CUST-0000005,Yug Gill,2025-10-16,1,2581.0,2581.0,NaN,2025-10-16,Delhi / Tier 1,Offline (Friends/Family),N
4,CUST-0000007,Daksha Ramaswamy,2024-03-11,2,5365.0,2682.5,303.0,2025-01-08,Bengaluru / Tier 1,Meta,Y


### Spending features:

In [7]:
spending = (
    orders
    .groupby("customer_id")
    .agg(
        total_spend=("net_value", "sum"),
        avg_spend=("net_value", "mean"),
        median_spend=("net_value", "median"),
        max_spend=("net_value", "max"),
        min_spend=("net_value", "min"),
        gross_revenue=("gross_value", "sum")
    )
    .reset_index()
)
spending.head()

,customer_id,total_spend,avg_spend,median_spend,max_spend,min_spend,gross_revenue
0,CUST-0000001,3356,3356.0,3356.0,3356,3356,3663
1,CUST-0000003,2360,2360.0,2360.0,2360,2360,2485
2,CUST-0000004,4029,2014.5,2014.5,2291,1738,4860
3,CUST-0000005,2581,2581.0,2581.0,2581,2581,3584
4,CUST-0000007,5365,2682.5,2682.5,2994,2371,6679


### Discount Features:

In [12]:
# Convert discount_percent to numeric
orders["discount_percent"] = (
    orders["discount_percent"]
    .astype(str)
    .str.replace("%", "", regex=False)
    .str.strip()
)

orders["discount_percent"] = pd.to_numeric(
    orders["discount_percent"],
    errors="coerce"
)

orders["discount_percent"] = orders["discount_percent"].fillna(0)

orders[["discount_amount", "discount_percent"]].dtypes

discount_amount       int64
discount_percent    float64
dtype: object

In [13]:
discount = (
    orders.groupby("customer_id")
    .agg(
        total_discount=("discount_amount", "sum"),
        avg_discount=("discount_amount", "mean"),
        avg_discount_percent=("discount_percent", "mean")
    )
    .reset_index()
)
discount.head()

,customer_id,total_discount,avg_discount,avg_discount_percent
0,CUST-0000001,307,307.0,8.40
1,CUST-0000003,125,125.0,5.00
2,CUST-0000004,832,416.0,17.50
3,CUST-0000005,1003,1003.0,28.00
4,CUST-0000007,1314,657.0,19.95


### Merge spending and discount features:

In [14]:
customer_features = (
    customer_features
    .merge(spending, on="customer_id", how="left")
    .merge(discount, on="customer_id", how="left")
)
customer_features.head()

,customer_id,name,first_order_date,total_orders,total_revenue,average_order_value,time_to_2nd_purchase,last_purchase_date,city_/_tier,acquisition_channel_(first_touch),repurchased,total_spend,avg_spend,median_spend,max_spend,min_spend,gross_revenue,total_discount,avg_discount,avg_discount_percent
0,CUST-0000001,Tejas Sarkar,2025-09-08,1,3356.0,3356.0,NaN,2025-09-08,Mumbai / Tier 1,Meta,N,3356,3356.0,3356.0,3356,3356,3663,307,307.0,8.40
1,CUST-0000003,Lila Bedi,2023-09-11,1,2360.0,2360.0,NaN,2023-09-11,Bengaluru / Tier 1,Google,N,2360,2360.0,2360.0,2360,2360,2485,125,125.0,5.00
2,CUST-0000004,Hamsini Parekh,2022-03-18,2,4029.0,2014.5,449.0,2023-06-11,Hyderabad / Tier 1,Google,Y,4029,2014.5,2014.5,2291,1738,4860,832,416.0,17.50
3,CUST-0000005,Yug Gill,2025-10-16,1,2581.0,2581.0,NaN,2025-10-16,Delhi / Tier 1,Offline (Friends/Family),N,2581,2581.0,2581.0,2581,2581,3584,1003,1003.0,28.00
4,CUST-0000007,Daksha Ramaswamy,2024-03-11,2,5365.0,2682.5,303.0,2025-01-08,Bengaluru / Tier 1,Meta,Y,5365,2682.5,2682.5,2994,2371,6679,1314,657.0,19.95


### Validation matrix:

In [15]:
print("Rows:",customer_features.shape[0])
print("Columns:",customer_features.shape[1])
customer_features.info()

Rows: 16865
Columns: 20
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 16865 entries, 0 to 16864
Data columns (total 20 columns):
 #   Column                             Non-Null Count  Dtype  
---  ------                             --------------  -----  
 0   customer_id                        16865 non-null  object 
 1   name                               16865 non-null  object 
 2   first_order_date                   16865 non-null  object 
 3   total_orders                       16865 non-null  int64  
 4   total_revenue                      16865 non-null  float64
 5   average_order_value                16865 non-null  float64
 6   time_to_2nd_purchase               7748 non-null   float64
 7   last_purchase_date                 16865 non-null  object 
 8   city_/_tier                        16865 non-null  object 
 9   acquisition_channel_(first_touch)  16865 non-null  object 
 10  repurchased                        16865 non-null  object 
 11  total_spend                   

### Order behavior features:

In [16]:
# Total completed, returned and RTO orders
order_behaviour = (
    orders.groupby("customer_id")
    .agg(
        total_orders_actual=("order_id", "count"),
        completed_orders=("order_status", lambda x: (x.str.lower() == "delivered").sum()),
        returned_orders=("order_status", lambda x: (x.str.lower() == "returned").sum()),
        rto_orders=("order_status", lambda x: (x.str.lower() == "rto").sum())
    )
    .reset_index()
)
order_behaviour.head()

,customer_id,total_orders_actual,completed_orders,returned_orders,rto_orders
0,CUST-0000001,1,0,1,0
1,CUST-0000003,1,1,0,0
2,CUST-0000004,2,0,2,0
3,CUST-0000005,1,1,0,0
4,CUST-0000007,2,0,1,1


### Merege order behavior:

In [17]:
customer_features = customer_features.merge(
    order_behaviour,
    on="customer_id",
    how="left"
)
customer_features.head()

,customer_id,name,first_order_date,total_orders,total_revenue,average_order_value,time_to_2nd_purchase,last_purchase_date,city_/_tier,acquisition_channel_(first_touch),repurchased,total_spend,avg_spend,median_spend,max_spend,min_spend,gross_revenue,total_discount,avg_discount,avg_discount_percent,total_orders_actual,completed_orders,returned_orders,rto_orders
0,CUST-0000001,Tejas Sarkar,2025-09-08,1,3356.0,3356.0,NaN,2025-09-08,Mumbai / Tier 1,Meta,N,3356,3356.0,3356.0,3356,3356,3663,307,307.0,8.40,1,0,1,0
1,CUST-0000003,Lila Bedi,2023-09-11,1,2360.0,2360.0,NaN,2023-09-11,Bengaluru / Tier 1,Google,N,2360,2360.0,2360.0,2360,2360,2485,125,125.0,5.00,1,1,0,0
2,CUST-0000004,Hamsini Parekh,2022-03-18,2,4029.0,2014.5,449.0,2023-06-11,Hyderabad / Tier 1,Google,Y,4029,2014.5,2014.5,2291,1738,4860,832,416.0,17.50,2,0,2,0
3,CUST-0000005,Yug Gill,2025-10-16,1,2581.0,2581.0,NaN,2025-10-16,Delhi / Tier 1,Offline (Friends/Family),N,2581,2581.0,2581.0,2581,2581,3584,1003,1003.0,28.00,1,1,0,0
4,CUST-0000007,Daksha Ramaswamy,2024-03-11,2,5365.0,2682.5,303.0,2025-01-08,Bengaluru / Tier 1,Meta,Y,5365,2682.5,2682.5,2994,2371,6679,1314,657.0,19.95,2,0,1,1


### Payment behavior:

In [18]:
payment_counts = (
    orders.groupby(["customer_id", "payment_mode"])
    .size()
    .reset_index(name="count")
)
preferred_payment = (
    payment_counts
    .sort_values(["customer_id", "count"], ascending=[True, False])
    .drop_duplicates("customer_id")
)
preferred_payment = preferred_payment[[
    "customer_id",
    "payment_mode"
]].rename(columns={
    "payment_mode": "preferred_payment"
})
preferred_payment.head()

,customer_id,preferred_payment
0,CUST-0000001,COD
1,CUST-0000003,COD
2,CUST-0000004,Card
3,CUST-0000005,COD
4,CUST-0000007,COD


### Diversity in payment:

In [19]:
payment_diversity = (
    orders.groupby("customer_id")
    .agg(
        payment_diversity=("payment_mode", "nunique")
    )
    .reset_index()
)
payment_diversity.head()

,customer_id,payment_diversity
0,CUST-0000001,1
1,CUST-0000003,1
2,CUST-0000004,1
3,CUST-0000005,1
4,CUST-0000007,2


### Merging payment features:

In [20]:
customer_features = (
    customer_features
    .merge(preferred_payment, on="customer_id", how="left")
    .merge(payment_diversity, on="customer_id", how="left")
)
customer_features.head()

,customer_id,name,first_order_date,total_orders,total_revenue,average_order_value,time_to_2nd_purchase,last_purchase_date,city_/_tier,acquisition_channel_(first_touch),repurchased,total_spend,avg_spend,median_spend,max_spend,min_spend,gross_revenue,total_discount,avg_discount,avg_discount_percent,total_orders_actual,completed_orders,returned_orders,rto_orders,preferred_payment,payment_diversity
0,CUST-0000001,Tejas Sarkar,2025-09-08,1,3356.0,3356.0,NaN,2025-09-08,Mumbai / Tier 1,Meta,N,3356,3356.0,3356.0,3356,3356,3663,307,307.0,8.40,1,0,1,0,COD,1
1,CUST-0000003,Lila Bedi,2023-09-11,1,2360.0,2360.0,NaN,2023-09-11,Bengaluru / Tier 1,Google,N,2360,2360.0,2360.0,2360,2360,2485,125,125.0,5.00,1,1,0,0,COD,1
2,CUST-0000004,Hamsini Parekh,2022-03-18,2,4029.0,2014.5,449.0,2023-06-11,Hyderabad / Tier 1,Google,Y,4029,2014.5,2014.5,2291,1738,4860,832,416.0,17.50,2,0,2,0,Card,1
3,CUST-0000005,Yug Gill,2025-10-16,1,2581.0,2581.0,NaN,2025-10-16,Delhi / Tier 1,Offline (Friends/Family),N,2581,2581.0,2581.0,2581,2581,3584,1003,1003.0,28.00,1,1,0,0,COD,1
4,CUST-0000007,Daksha Ramaswamy,2024-03-11,2,5365.0,2682.5,303.0,2025-01-08,Bengaluru / Tier 1,Meta,Y,5365,2682.5,2682.5,2994,2371,6679,1314,657.0,19.95,2,0,1,1,COD,2


### Product behavior:

In [21]:
product_features = (
    master.groupby("customer_id")
    .agg(
        category_diversity=("category", "nunique"),
        size_diversity=("size", "nunique"),
        color_diversity=("color", "nunique"),
        vendor_diversity=("vendor", "nunique")
    )
    .reset_index()
)
product_features.head()

,customer_id,category_diversity,size_diversity,color_diversity,vendor_diversity
0,CUST-0000001,2,2,2,2
1,CUST-0000003,1,1,1,1
2,CUST-0000004,2,3,3,3
3,CUST-0000005,2,1,1,2
4,CUST-0000007,3,3,2,3


### Fav product category:

In [22]:
favorite_category = (
    master.groupby(["customer_id", "category"])
    .size()
    .reset_index(name="count")
)

favorite_category = (
    favorite_category
    .sort_values(["customer_id", "count"], ascending=[True, False])
    .drop_duplicates("customer_id")
)

favorite_category = favorite_category[[
    "customer_id",
    "category"
]].rename(columns={
    "category": "favorite_category"
})

favorite_category.head()

,customer_id,favorite_category
0,CUST-0000001,shirts
2,CUST-0000003,pants
3,CUST-0000004,pants
5,CUST-0000005,croptops
7,CUST-0000007,croptops


### Return behavior:

In [23]:
returns = (
    master.groupby("customer_id")
    .agg(
        returned_items=("returned", lambda x: (x == "Y").sum()),
        total_items=("returned", "count")
    )
    .reset_index()
)

returns["return_rate"] = (
    returns["returned_items"] /
    returns["total_items"]
)

returns.head()

,customer_id,returned_items,total_items,return_rate
0,CUST-0000001,2,2,1.0
1,CUST-0000003,0,1,0.0
2,CUST-0000004,3,3,1.0
3,CUST-0000005,0,2,0.0
4,CUST-0000007,3,3,1.0


### Merging product features:

In [24]:
customer_features = (
    customer_features
    .merge(product_features, on="customer_id", how="left")
    .merge(favorite_category, on="customer_id", how="left")
    .merge(returns[[
        "customer_id",
        "return_rate"
    ]], on="customer_id", how="left")
)
customer_features.head()

,customer_id,name,first_order_date,total_orders,total_revenue,average_order_value,time_to_2nd_purchase,last_purchase_date,city_/_tier,acquisition_channel_(first_touch),repurchased,total_spend,avg_spend,median_spend,max_spend,min_spend,gross_revenue,total_discount,avg_discount,avg_discount_percent,total_orders_actual,completed_orders,returned_orders,rto_orders,preferred_payment,payment_diversity,category_diversity,size_diversity,color_diversity,vendor_diversity,favorite_category,return_rate
0,CUST-0000001,Tejas Sarkar,2025-09-08,1,3356.0,3356.0,NaN,2025-09-08,Mumbai / Tier 1,Meta,N,3356,3356.0,3356.0,3356,3356,3663,307,307.0,8.40,1,0,1,0,COD,1,2,2,2,2,shirts,1.0
1,CUST-0000003,Lila Bedi,2023-09-11,1,2360.0,2360.0,NaN,2023-09-11,Bengaluru / Tier 1,Google,N,2360,2360.0,2360.0,2360,2360,2485,125,125.0,5.00,1,1,0,0,COD,1,1,1,1,1,pants,0.0
2,CUST-0000004,Hamsini Parekh,2022-03-18,2,4029.0,2014.5,449.0,2023-06-11,Hyderabad / Tier 1,Google,Y,4029,2014.5,2014.5,2291,1738,4860,832,416.0,17.50,2,0,2,0,Card,1,2,3,3,3,pants,1.0
3,CUST-0000005,Yug Gill,2025-10-16,1,2581.0,2581.0,NaN,2025-10-16,Delhi / Tier 1,Offline (Friends/Family),N,2581,2581.0,2581.0,2581,2581,3584,1003,1003.0,28.00,1,1,0,0,COD,1,2,1,1,2,croptops,0.0
4,CUST-0000007,Daksha Ramaswamy,2024-03-11,2,5365.0,2682.5,303.0,2025-01-08,Bengaluru / Tier 1,Meta,Y,5365,2682.5,2682.5,2994,2371,6679,1314,657.0,19.95,2,0,1,1,COD,2,3,3,2,3,croptops,1.0


### Feature matrix:

In [25]:
print("Rows:",customer_features.shape[0])
print("Columns:",customer_features.shape[1])
customer_features.head()

Rows: 16865
Columns: 32


,customer_id,name,first_order_date,total_orders,total_revenue,average_order_value,time_to_2nd_purchase,last_purchase_date,city_/_tier,acquisition_channel_(first_touch),repurchased,total_spend,avg_spend,median_spend,max_spend,min_spend,gross_revenue,total_discount,avg_discount,avg_discount_percent,total_orders_actual,completed_orders,returned_orders,rto_orders,preferred_payment,payment_diversity,category_diversity,size_diversity,color_diversity,vendor_diversity,favorite_category,return_rate
0,CUST-0000001,Tejas Sarkar,2025-09-08,1,3356.0,3356.0,NaN,2025-09-08,Mumbai / Tier 1,Meta,N,3356,3356.0,3356.0,3356,3356,3663,307,307.0,8.40,1,0,1,0,COD,1,2,2,2,2,shirts,1.0
1,CUST-0000003,Lila Bedi,2023-09-11,1,2360.0,2360.0,NaN,2023-09-11,Bengaluru / Tier 1,Google,N,2360,2360.0,2360.0,2360,2360,2485,125,125.0,5.00,1,1,0,0,COD,1,1,1,1,1,pants,0.0
2,CUST-0000004,Hamsini Parekh,2022-03-18,2,4029.0,2014.5,449.0,2023-06-11,Hyderabad / Tier 1,Google,Y,4029,2014.5,2014.5,2291,1738,4860,832,416.0,17.50,2,0,2,0,Card,1,2,3,3,3,pants,1.0
3,CUST-0000005,Yug Gill,2025-10-16,1,2581.0,2581.0,NaN,2025-10-16,Delhi / Tier 1,Offline (Friends/Family),N,2581,2581.0,2581.0,2581,2581,3584,1003,1003.0,28.00,1,1,0,0,COD,1,2,1,1,2,croptops,0.0
4,CUST-0000007,Daksha Ramaswamy,2024-03-11,2,5365.0,2682.5,303.0,2025-01-08,Bengaluru / Tier 1,Meta,Y,5365,2682.5,2682.5,2994,2371,6679,1314,657.0,19.95,2,0,1,1,COD,2,3,3,2,3,croptops,1.0


### Website behavior features:

In [26]:
website_features = (
    website_sessions.groupby("customer_id")
    .agg(
        total_sessions=("sessions", "sum"),
        total_product_views=("product_views", "sum"),
        total_add_to_cart=("add_to_cart", "sum"),
        total_checkout=("begin_checkout", "sum"),
        total_purchases=("purchased", "sum"),
        website_revenue=("revenue", "sum")
    )
    .reset_index()
)
website_features.head()

,customer_id,total_sessions,total_product_views,total_add_to_cart,total_checkout,total_purchases,website_revenue
0,CUST-0000001,1,4,1,1,1,3356.0
1,CUST-0000004,4,11,4,4,4,8058.0
2,CUST-0000007,2,4,2,2,2,5365.0
3,CUST-0000009,3,7,3,3,3,4871.0
4,CUST-0000012,1,1,1,1,1,3215.0


### conversion matrix:

In [27]:
website_features["view_to_cart_rate"] = (
    website_features["total_add_to_cart"] /
    website_features["total_product_views"]
)

website_features["cart_to_checkout_rate"] = (
    website_features["total_checkout"] /
    website_features["total_add_to_cart"]
)

website_features["checkout_to_purchase_rate"] = (
    website_features["total_purchases"] /
    website_features["total_checkout"]
)

website_features["revenue_per_session"] = (
    website_features["website_revenue"] /
    website_features["total_sessions"]
)

website_features.replace([np.inf, -np.inf], np.nan, inplace=True)
website_features.fillna(0, inplace=True)

website_features.head()

,customer_id,total_sessions,total_product_views,total_add_to_cart,total_checkout,total_purchases,website_revenue,view_to_cart_rate,cart_to_checkout_rate,checkout_to_purchase_rate,revenue_per_session
0,CUST-0000001,1,4,1,1,1,3356.0,0.250000,1.0,1.0,3356.000000
1,CUST-0000004,4,11,4,4,4,8058.0,0.363636,1.0,1.0,2014.500000
2,CUST-0000007,2,4,2,2,2,5365.0,0.500000,1.0,1.0,2682.500000
3,CUST-0000009,3,7,3,3,3,4871.0,0.428571,1.0,1.0,1623.666667
4,CUST-0000012,1,1,1,1,1,3215.0,1.000000,1.0,1.0,3215.000000


### Customer device preference:

In [28]:
device = (
    website_sessions.groupby(["customer_id", "device_category"])
    .size()
    .reset_index(name="count")
)

device = (
    device.sort_values(
        ["customer_id", "count"],
        ascending=[True, False]
    )
    .drop_duplicates("customer_id")
)

device = device.rename(
    columns={
        "device_category": "preferred_device"
    }
)[["customer_id", "preferred_device"]]

device.head()

,customer_id,preferred_device
0,CUST-0000001,mobile
1,CUST-0000004,desktop
3,CUST-0000007,mobile
5,CUST-0000009,mobile
6,CUST-0000012,mobile


### Traffic source preferences:

In [29]:
traffic = (
    website_sessions.groupby(
        ["customer_id", "traffic_source"]
    )
    .size()
    .reset_index(name="count")
)

traffic = (
    traffic.sort_values(
        ["customer_id", "count"],
        ascending=[True, False]
    )
    .drop_duplicates("customer_id")
)

traffic = traffic.rename(
    columns={
        "traffic_source": "preferred_traffic_source"
    }
)[["customer_id", "preferred_traffic_source"]]

traffic.head()

,customer_id,preferred_traffic_source
0,CUST-0000001,Meta
1,CUST-0000004,Google
3,CUST-0000007,Meta
5,CUST-0000009,Meta
6,CUST-0000012,Meta


### Campaige preferences:

In [30]:
campaign = (
    website_sessions.groupby(
        ["customer_id", "campaign_name"]
    )
    .size()
    .reset_index(name="count")
)

campaign = (
    campaign.sort_values(
        ["customer_id", "count"],
        ascending=[True, False]
    )
    .drop_duplicates("customer_id")
)

campaign = campaign.rename(
    columns={
        "campaign_name": "favorite_campaign"
    }
)[["customer_id", "favorite_campaign"]]

campaign.head()

,customer_id,favorite_campaign
0,CUST-0000001,KS_Sep25_Camp_46
1,CUST-0000004,KS_AlwaysOn
2,CUST-0000007,KS_AlwaysOn
4,CUST-0000009,KS_AlwaysOn
5,CUST-0000012,KS_Sep25_Camp_45


### Merging website features:

customer_features = (
    customer_features
    .merge(website_features,
           on="customer_id",
           how="left")
    .merge(device,
           on="customer_id",
           how="left")
    .merge(traffic,
           on="customer_id",
           how="left")
    .merge(campaign,
           on="customer_id",
           how="left")
)
customer_features.head()

### Customer lifetime value:

In [32]:
customer_features["customer_lifetime_value"] = (
    customer_features["average_order_value"] *
    customer_features["total_orders"]
)

### Customer loyalty score:

In [33]:
customer_features["loyalty_score"] = (
    customer_features["total_orders"] *
    customer_features["payment_diversity"]
)

### Missing value checking:

In [34]:
numeric_cols = customer_features.select_dtypes(
    include=np.number
).columns

customer_features[numeric_cols] = (
    customer_features[numeric_cols]
    .fillna(0)
)

categorical_cols = customer_features.select_dtypes(
    include="object"
).columns

customer_features[categorical_cols] = (
    customer_features[categorical_cols]
    .fillna("Unknown")
)

### Fill missing values:

In [35]:
numeric_cols = customer_features.select_dtypes(
    include=np.number
).columns

customer_features[numeric_cols] = (
    customer_features[numeric_cols]
    .fillna(0)
)

categorical_cols = customer_features.select_dtypes(
    include="object"
).columns

customer_features[categorical_cols] = (
    customer_features[categorical_cols]
    .fillna("Unknown")
)

### Feature matrix:

In [37]:
print("Feature Matrix Shape:",customer_features.shape)
customer_features.head()

Feature Matrix Shape: (16865, 47)


,customer_id,name,first_order_date,total_orders,total_revenue,average_order_value,time_to_2nd_purchase,last_purchase_date,city_/_tier,acquisition_channel_(first_touch),repurchased,total_spend,avg_spend,median_spend,max_spend,min_spend,gross_revenue,total_discount,avg_discount,avg_discount_percent,total_orders_actual,completed_orders,returned_orders,rto_orders,preferred_payment,payment_diversity,category_diversity,size_diversity,color_diversity,vendor_diversity,favorite_category,return_rate,total_sessions,total_product_views,total_add_to_cart,total_checkout,total_purchases,website_revenue,view_to_cart_rate,cart_to_checkout_rate,checkout_to_purchase_rate,revenue_per_session,preferred_device,preferred_traffic_source,favorite_campaign,customer_lifetime_value,loyalty_score
0,CUST-0000001,Tejas Sarkar,2025-09-08,1,3356.0,3356.0,0.0,2025-09-08,Mumbai / Tier 1,Meta,N,3356,3356.0,3356.0,3356,3356,3663,307,307.0,8.40,1,0,1,0,COD,1,2,2,2,2,shirts,1.0,1.0,4.0,1.0,1.0,1.0,3356.0,0.250000,1.0,1.0,3356.0,mobile,Meta,KS_Sep25_Camp_46,3356.0,1
1,CUST-0000003,Lila Bedi,2023-09-11,1,2360.0,2360.0,0.0,2023-09-11,Bengaluru / Tier 1,Google,N,2360,2360.0,2360.0,2360,2360,2485,125,125.0,5.00,1,1,0,0,COD,1,1,1,1,1,pants,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,Unknown,Unknown,Unknown,2360.0,1
2,CUST-0000004,Hamsini Parekh,2022-03-18,2,4029.0,2014.5,449.0,2023-06-11,Hyderabad / Tier 1,Google,Y,4029,2014.5,2014.5,2291,1738,4860,832,416.0,17.50,2,0,2,0,Card,1,2,3,3,3,pants,1.0,4.0,11.0,4.0,4.0,4.0,8058.0,0.363636,1.0,1.0,2014.5,desktop,Google,KS_AlwaysOn,4029.0,2
3,CUST-0000005,Yug Gill,2025-10-16,1,2581.0,2581.0,0.0,2025-10-16,Delhi / Tier 1,Offline (Friends/Family),N,2581,2581.0,2581.0,2581,2581,3584,1003,1003.0,28.00,1,1,0,0,COD,1,2,1,1,2,croptops,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,Unknown,Unknown,Unknown,2581.0,1
4,CUST-0000007,Daksha Ramaswamy,2024-03-11,2,5365.0,2682.5,303.0,2025-01-08,Bengaluru / Tier 1,Meta,Y,5365,2682.5,2682.5,2994,2371,6679,1314,657.0,19.95,2,0,1,1,COD,2,3,3,2,3,croptops,1.0,2.0,4.0,2.0,2.0,2.0,5365.0,0.500000,1.0,1.0,2682.5,mobile,Meta,KS_AlwaysOn,5365.0,4


### Repurchase flag:

In [39]:
customer_features["repurchased"] = (
    customer_features["repurchased"]
    .map({
        "Y": 1,
        "N": 0
    })
)
customer_features["repurchased"] = (
    customer_features["repurchased"]
    .fillna(0)
    .astype(int)
)

### Customer value segment:

In [40]:
customer_features["value_segment"] = pd.qcut(
    customer_features["total_revenue"],
    q=3,
    labels=[
        "Low",
        "Medium",
        "High"
    ],
    duplicates="drop"
)

customer_features[[
    "total_revenue",
    "value_segment"
]].head()

,total_revenue,value_segment
0,3356.0,Medium
1,2360.0,Low
2,4029.0,Medium
3,2581.0,Low
4,5365.0,High


### Summary:

In [41]:
print("=" * 60)
print("Customer Feature Matrix")
print("=" * 60)

print(f"Rows    : {customer_features.shape[0]}")
print(f"Columns : {customer_features.shape[1]}")

customer_features.info()

Customer Feature Matrix
Rows    : 16865
Columns : 48
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 16865 entries, 0 to 16864
Data columns (total 48 columns):
 #   Column                             Non-Null Count  Dtype   
---  ------                             --------------  -----   
 0   customer_id                        16865 non-null  object  
 1   name                               16865 non-null  object  
 2   first_order_date                   16865 non-null  object  
 3   total_orders                       16865 non-null  int64   
 4   total_revenue                      16865 non-null  float64 
 5   average_order_value                16865 non-null  float64 
 6   time_to_2nd_purchase               16865 non-null  float64 
 7   last_purchase_date                 16865 non-null  object  
 8   city_/_tier                        16865 non-null  object  
 9   acquisition_channel_(first_touch)  16865 non-null  object  
 10  repurchased                        16865 non-null  in

### Missing values:

In [42]:
missing = customer_features.isnull().sum()
missing = missing[missing > 0]

print("Missing Values")

missing.sort_values(ascending=False)

Missing Values


Series([], dtype: int64)

### Saving features matrix:

In [43]:
OUTPUT = Path("data/processed")
customer_features.to_csv(
    OUTPUT / "customer_features.csv",
    index=False
)
print("customer_features.csv saved successfully..!")

customer_features.csv saved successfully..!


In [45]:
print(f"Total Customers : {customer_features.shape[0]}")
print(f"Total Features  : {customer_features.shape[1]}")

print("\nSample Data")

customer_features.head()

Total Customers : 16865
Total Features  : 48

Sample Data


,customer_id,name,first_order_date,total_orders,total_revenue,average_order_value,time_to_2nd_purchase,last_purchase_date,city_/_tier,acquisition_channel_(first_touch),repurchased,total_spend,avg_spend,median_spend,max_spend,min_spend,gross_revenue,total_discount,avg_discount,avg_discount_percent,total_orders_actual,completed_orders,returned_orders,rto_orders,preferred_payment,payment_diversity,category_diversity,size_diversity,color_diversity,vendor_diversity,favorite_category,return_rate,total_sessions,total_product_views,total_add_to_cart,total_checkout,total_purchases,website_revenue,view_to_cart_rate,cart_to_checkout_rate,checkout_to_purchase_rate,revenue_per_session,preferred_device,preferred_traffic_source,favorite_campaign,customer_lifetime_value,loyalty_score,value_segment
0,CUST-0000001,Tejas Sarkar,2025-09-08,1,3356.0,3356.0,0.0,2025-09-08,Mumbai / Tier 1,Meta,0,3356,3356.0,3356.0,3356,3356,3663,307,307.0,8.40,1,0,1,0,COD,1,2,2,2,2,shirts,1.0,1.0,4.0,1.0,1.0,1.0,3356.0,0.250000,1.0,1.0,3356.0,mobile,Meta,KS_Sep25_Camp_46,3356.0,1,Medium
1,CUST-0000003,Lila Bedi,2023-09-11,1,2360.0,2360.0,0.0,2023-09-11,Bengaluru / Tier 1,Google,0,2360,2360.0,2360.0,2360,2360,2485,125,125.0,5.00,1,1,0,0,COD,1,1,1,1,1,pants,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,Unknown,Unknown,Unknown,2360.0,1,Low
2,CUST-0000004,Hamsini Parekh,2022-03-18,2,4029.0,2014.5,449.0,2023-06-11,Hyderabad / Tier 1,Google,1,4029,2014.5,2014.5,2291,1738,4860,832,416.0,17.50,2,0,2,0,Card,1,2,3,3,3,pants,1.0,4.0,11.0,4.0,4.0,4.0,8058.0,0.363636,1.0,1.0,2014.5,desktop,Google,KS_AlwaysOn,4029.0,2,Medium
3,CUST-0000005,Yug Gill,2025-10-16,1,2581.0,2581.0,0.0,2025-10-16,Delhi / Tier 1,Offline (Friends/Family),0,2581,2581.0,2581.0,2581,2581,3584,1003,1003.0,28.00,1,1,0,0,COD,1,2,1,1,2,croptops,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,Unknown,Unknown,Unknown,2581.0,1,Low
4,CUST-0000007,Daksha Ramaswamy,2024-03-11,2,5365.0,2682.5,303.0,2025-01-08,Bengaluru / Tier 1,Meta,1,5365,2682.5,2682.5,2994,2371,6679,1314,657.0,19.95,2,0,1,1,COD,2,3,3,2,3,croptops,1.0,2.0,4.0,2.0,2.0,2.0,5365.0,0.500000,1.0,1.0,2682.5,mobile,Meta,KS_AlwaysOn,5365.0,4,High
